## TEST

In [1]:
import sys, torch, os

print("Python:", sys.version)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
print("cwd:", os.getcwd())

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
CUDA: True
GPU: Tesla T4
cwd: /content


# 04A — Frozen Log-Mel training

Four from-scratch classifiers in a controlled comparison of two frozen acoustic frontend configurations.
No frozen data/frontend artifact is rewritten. Read [the full protocol](../../docs/STAGE04.md) before training.
Install `requirements-stage04.txt` in a dedicated CPython 3.12 kernel. Training is deliberately opt-in below;
all preflight checks run first. This notebook uses shared code with its companion branch.


In [2]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "tools/training_common.py").is_file())
STAGE04_PYTHON = PROJECT / ".venv-stage04" / "Scripts" / "python.exe"
if sys.version_info[:2] != (3, 12) or Path(sys.prefix).resolve() != (PROJECT / ".venv-stage04").resolve():
    raise RuntimeError(
        f"Wrong Stage-04 kernel: {sys.executable} (Python {sys.version.split()[0]}). "
        f"Select Kernel -> Python Environments -> {STAGE04_PYTHON}, "
        "then restart the kernel and Run All. The scientific AST locks require the "
        "accepted Python 3.12 environment; another Python version can report changed "
        "definitions even when the frozen source is unchanged. Do not update the locks."
    )
print(f"Stage-04 kernel: {sys.executable} (Python {sys.version.split()[0]})")
sys.path.insert(0, str(PROJECT / "tools"))
import importlib
import stage04_data, model_zoo, training_common
for module in (stage04_data, model_zoo, training_common):
    importlib.reload(module)
import pandas as pd
from IPython.display import display
from model_zoo import FAMILIES, architecture_config, build_model
from training_common import CONFIG, run_branch, compare_saved_results, RUNS
from stage04_data import BRANCHES, CLASSES, ROOT, prepare_caches, read_json

FRONTEND = "03A"
RUN_TRAINING = True


StopIteration: 

## Fixed protocol and architecture definitions

CPU PyTorch; seed 42; TRAIN-only inverse-frequency weights; weighted cross-entropy; Adam 0.001;
batch 64; constant learning rate; at most 50 epochs; patience 8; lowest weighted validation loss
selects the checkpoint (earliest exact tie). No input normalization or augmentation.
TEST is evaluated only after the selected checkpoint and its SHA256 are saved.


In [6]:
display(pd.Series(CONFIG, name="Fixed Stage-04 protocol"))
architecture_rows = []
for family in FAMILIES:
    width = BRANCHES[FRONTEND]["shape"][1]
    model = build_model(family, width)
    architecture_rows.append({"family": family, "input": [96, width],
                              "parameters": sum(p.numel() for p in model.parameters())})
    print(family, architecture_config(family, width))
display(pd.DataFrame(architecture_rows))


seed                                                                            42
device                                                                         cpu
dtype                                                                      float32
cpu_threads                                                                      4
batch_size                                                                      64
max_epochs                                                                      50
early_stopping_patience                                                          8
selection                                minimum_validation_weighted_cross_entropy
min_delta                                                                      0.0
tie_rule                                                            earliest_epoch
optimizer                                                                     Adam
learning_rate                                                                0.001
beta

small_cnn {'channels': [16, 32, 64], 'kernel': [3, 3], 'stride': [1, 1], 'max_pool_after_first_two': [2, 2], 'head': 'global_mean_time_frequency -> dropout -> linear4', 'family': 'small_cnn', 'input_shape': [1, 96, 64], 'classes': 4, 'activation': 'ReLU', 'dropout': 0.2, 'convolution_bias': False, 'convolution_padding': 'symmetric k//2', 'batch_norm': {'eps': 1e-05, 'momentum': 0.1, 'affine': True}, 'linear_bias': True, 'initialization': 'PyTorch default reset_parameters; seed 42', 'pretrained': False}
ds_cnn {'stem_channels': 32, 'stem_kernel': [3, 3], 'stem_stride': [2, 2], 'blocks': 4, 'channels': 32, 'depthwise_kernel': [3, 3], 'block_stride': [1, 1], 'pointwise_kernel': [1, 1], 'head': 'global_mean_time_frequency -> dropout -> linear4', 'family': 'ds_cnn', 'input_shape': [1, 96, 64], 'classes': 4, 'activation': 'ReLU', 'dropout': 0.2, 'convolution_bias': False, 'convolution_padding': 'symmetric k//2', 'batch_norm': {'eps': 1e-05, 'momentum': 0.1, 'affine': True}, 'linear_bias': Tr

,family,input,parameters
0,small_cnn,"[96, 64]",23668
1,ds_cnn,"[96, 64]",6244
2,tc_resnet8,"[96, 64]",65940
3,cnn_lstm,"[96, 64]",13428


## Mandatory preflight and paired caches

Authenticates all accepted data/source/WAV identities, frozen audits, labels, complete grid and split membership.
Builds separate Stage-04 caches if missing; checks every cached tensor and exact example alignment across both
branches before training. Do not repair failures by updating frozen pins, dropping examples or changing DSP.
Historical 00–02 notebook execution-hash discrepancies, if present, are reported explicitly; see protocol notes.


In [7]:
caches, evidence = prepare_caches()
display(pd.DataFrame({"species": CLASSES, "TRAIN examples": evidence["train_counts"],
                     "class weight": [evidence["class_weights"][s] for s in CLASSES]}))
display(pd.Series(evidence["counts"], name="Frozen split counts"))
for note in evidence["historical_review_notes"]:
    print(note)
assert evidence["identical_branch_example_uids_labels_splits"]
print("PASS: paired caches, 32,645 identities, four labels, fixed splits and no source-name leakage.")
del caches  # run_branch independently authenticates and opens its own read-only maps


,species,TRAIN examples,class weight
0,an arabiensis,10457,0.540619
1,an funestus ss,5128,1.102428
2,an squamosus,1426,3.964411
3,culex pipiens complex,5602,1.009149


train         22613
validation     4937
test           5095
Name: Frozen split counts, dtype: int64

{'notebook': 'notebooks/current/00_prepare_dataset.ipynb', 'historical_sha256': '2ee89d7b424c517b165a0c9a56f785d6540df79bd487abd1bd8be9c922cc95d3', 'current_frozen_sha256': '0d6115c2c04daa6922011fc25a6a2fcf0c40404f960db69da84244e0d1ba2478', 'note': 'Historical whole-notebook hash differs; handled as provenance only. Scientific AST/artifact gates are checked separately.'}
{'notebook': 'notebooks/current/01_dataset_audit.ipynb', 'historical_sha256': 'd8fdb548b5c3c6b4ce6a0d13723da692358adee77f49b709dddd79a7ab6babf2', 'current_frozen_sha256': 'fa110b1d4abf15fec9b5e34b091bda71867e82b0654a62060eff28a0ac47229e', 'note': 'Historical whole-notebook hash differs; handled as provenance only. Scientific AST/artifact gates are checked separately.'}
{'notebook': 'notebooks/current/02_segmentation.ipynb', 'historical_sha256': '05b65e69d225133441cce7ebf123124f7c4080e671a252c3fcb63e08febeb495', 'current_frozen_sha256': 'f22aede332ac8d6898cfcdecd82abf9bce17113b5ff18605c4430fd4c70242fc', 'note': 'Histori

## Execute four full experiments

Set `RUN_TRAINING = True` in the setup cell, then rerun the training cell when ready.
Do not alter the protocol for one frontend. Completed runs are authenticated and reused;
interrupted runs remain visible and require investigation. Training does not receive TEST loaders.


In [ ]:
if RUN_TRAINING:
    branch_results = run_branch(FRONTEND)
    display(branch_results)
else:
    print("READY; full training was not launched (RUN_TRAINING=False).")


epoch 01: train loss=1.221273, val loss=1.156333, val macro-F1=0.389467; best=1
epoch 02: train loss=1.102543, val loss=1.232946, val macro-F1=0.390058; best=1
epoch 03: train loss=1.029114, val loss=1.973723, val macro-F1=0.243184; best=1
epoch 04: train loss=0.985035, val loss=1.021740, val macro-F1=0.525004; best=4
epoch 05: train loss=0.968069, val loss=2.093409, val macro-F1=0.166227; best=4
epoch 06: train loss=0.946762, val loss=2.128974, val macro-F1=0.256893; best=4
epoch 07: train loss=0.926757, val loss=1.945441, val macro-F1=0.213328; best=4
epoch 08: train loss=0.913530, val loss=2.375197, val macro-F1=0.314101; best=4
epoch 09: train loss=0.899792, val loss=1.131867, val macro-F1=0.504983; best=4
epoch 10: train loss=0.881914, val loss=3.026249, val macro-F1=0.208265; best=4
epoch 11: train loss=0.870248, val loss=2.220551, val macro-F1=0.288941; best=4
epoch 12: train loss=0.852469, val loss=2.002911, val macro-F1=0.348988; best=4
epoch 01: train loss=1.217486, val loss=

## Interpretation

Primary results are example-level classification. Windows from a recording are correlated and are not
independent biological mosquitoes. Secondary source metrics average window probabilities by source name.
Use 04C only after both branches finish. Model size is serialized PyTorch size, not hardware Flash/RAM.
Stage 05 will assess quantization, latency and hardware compatibility; this notebook does none of those.
